In [1]:
import numpy as np
import pandas as pd

df_data = pd.read_csv('data/annotated_conflicts.csv')
df_merge = pd.read_csv('data/annotated_merged.csv')

In [2]:
from sklearn.metrics import cohen_kappa_score                                                                                                               
from itertools import combinations                                                                                                                          
                                                                                                                                                            
annotators = ['bias_label_praveen', 'bias_label_ashini', 'bias_label_dinithi']                                                                              
                                                                                                                                                            
# Pairwise Cohen's Kappa                                                                                                                                    
print("=== Pairwise Cohen's Kappa ===")                                                                                                                     
for a1, a2 in combinations(annotators, 2):                                                                                                                  
    mask = df_data[a1].notna() & df_data[a2].notna()                                                                                                        
    subset = df_data[mask]                                                                                                                                  
    kappa = cohen_kappa_score(subset[a1], subset[a2])                                                                                                       
    agreement = (subset[a1] == subset[a2]).mean() * 100                                                                                                     
    name1, name2 = a1.split('_')[-1], a2.split('_')[-1]                                                                                                     
    print(f"{name1} vs {name2}: Kappa = {kappa:.4f}, Agreement = {agreement:.2f}% (n={len(subset)})")                                                       
                                                                                                                                                            
# Fleiss' Kappa (all 3 annotators)                                                                                                                          
print("\n=== Fleiss' Kappa (all 3 annotators) ===")                                                                                                         
mask_all = df_data[annotators].notna().all(axis=1)                                                                                                          
df_all = df_data.loc[mask_all, annotators]                                                                                                                  
categories = sorted(df_all.values.flatten().tolist())                                                                                                       
categories = sorted(set(categories))                                                                                                                        
                                                                                                                                                            
n_items = len(df_all)                                                                                                                                       
n_raters = 3                                                                                                                                                
n_cat = len(categories)                                                                                                                                     

# Build category count matrix
cat_to_idx = {c: i for i, c in enumerate(categories)}
counts = np.zeros((n_items, n_cat))
for i, (_, row) in enumerate(df_all.iterrows()):
    for col in annotators:
        counts[i, cat_to_idx[row[col]]] += 1

P_i = (np.sum(counts ** 2, axis=1) - n_raters) / (n_raters * (n_raters - 1))
P_bar = np.mean(P_i)

p_j = np.sum(counts, axis=0) / (n_items * n_raters)
P_e = np.sum(p_j ** 2)

fleiss_kappa = (P_bar - P_e) / (1 - P_e) if P_e != 1 else 1.0
print(f"Fleiss' Kappa: {fleiss_kappa:.4f} (n={n_items})")
print(f"Overall agreement: {P_bar:.4f}")
print(f"Categories: {categories}")

# Distribution of labels per annotator
print("\n=== Label Distribution per Annotator ===")
for col in annotators:
    name = col.split('_')[-1]
    print(f"\n{name}:")
    print(df_data[col].value_counts(dropna=True).to_string())


=== Pairwise Cohen's Kappa ===
praveen vs ashini: Kappa = 0.2542, Agreement = 45.00% (n=20)
praveen vs dinithi: Kappa = 0.0964, Agreement = 33.33% (n=15)
ashini vs dinithi: Kappa = 0.3187, Agreement = 52.63% (n=19)

=== Fleiss' Kappa (all 3 annotators) ===
Fleiss' Kappa: 0.2315 (n=8)
Overall agreement: 0.4583
Categories: ['Center', 'Far Left', 'Far Right', 'Left', 'Right']

=== Label Distribution per Annotator ===

praveen:
bias_label_praveen
Center       9
Left         6
Right        6
Far Right    4
Far Left     2

ashini:
bias_label_ashini
Center       11
Left          9
Right         7
Far Right     3
Far Left      1

dinithi:
bias_label_dinithi
Center      14
Left         5
Far Left     4
Right        3


In [3]:
from scipy.stats import pearsonr

# Map labels to numeric values
label_map = {'Far Left': -2, 'Left': -1, 'Center': 0, 'Right': 1, 'Far Right': 2}

print("=== Pearson's Correlation Coefficient ===")
for a1, a2 in combinations(annotators, 2):
    mask = df_data[a1].notna() & df_data[a2].notna()
    subset = df_data[mask]
    vals1 = subset[a1].map(label_map)
    vals2 = subset[a2].map(label_map)
    corr, p_value = pearsonr(vals1, vals2)
    name1, name2 = a1.split('_')[-1], a2.split('_')[-1]
    print(f"{name1} vs {name2}: r = {corr:.4f}, p = {p_value:.4e} (n={len(subset)})")



=== Pearson's Correlation Coefficient ===
praveen vs ashini: r = -0.0702, p = 7.6881e-01 (n=20)
praveen vs dinithi: r = -0.2323, p = 4.0485e-01 (n=15)
ashini vs dinithi: r = 0.1083, p = 6.5911e-01 (n=19)


In [8]:
#Data set description
df_merge.info()

#Bias Label Distribution in Merged Data
df_merge['bias_label'].value_counts(dropna=False)


<class 'pandas.DataFrame'>
RangeIndex: 811 entries, 0 to 810
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   article_id    811 non-null    str  
 1   publisher     811 non-null    str  
 2   url           811 non-null    str  
 3   published_at  811 non-null    str  
 4   title         811 non-null    str  
 5   body_text     811 non-null    str  
 6   bias_label    811 non-null    str  
 7   annotator     811 non-null    str  
dtypes: str(8)
memory usage: 50.8 KB


bias_label
Center       309
Left         220
Right        173
Far Left      66
Far Right     43
Name: count, dtype: int64

### Iteration 1: Using Sinbert-Small 

In [11]:
df = df_merge[['article_id','title', 'body_text', 'bias_label']].copy()
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 811 entries, 0 to 810
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   article_id  811 non-null    str  
 1   title       811 non-null    str  
 2   body_text   811 non-null    str  
 3   bias_label  811 non-null    str  
dtypes: str(4)
memory usage: 25.5 KB


In [13]:
#Unicode normalization (NFC - canonical decomposition + composition)
import unicodedata                                                                                                                                          
                                                                                                                                                        
# Unicode normalization (NFC - canonical decomposition + composition)                                                                                       
df['body_text'] = df['body_text'].apply(lambda x: unicodedata.normalize('NFC', x))                                                                        
df['title'] = df['title'].apply(lambda x: unicodedata.normalize('NFC', x))

df.head()


,article_id,title,body_text,bias_label
0,005336e4-16fd-5a46-8eaf-aa3ef9ee9032,ලංකාවේ පළමු ස්වයංක්‍රීය දේශසීමා පාලන පද්ධතිය ව...,ලංකාවේ පළමු ස්වයංක්‍රීය දේශසීමා පාලන පද්ධතිය ව...,Left
1,01338392-d1a3-502f-bf49-9d1ee204ed22,අයිස් මත්ද්‍රව්‍ය කිලෝ 400ක් සමග කොටු වූ සැකකර...,අයිස් මත්ද්‍රව්‍ය කිලෝ 400ක් සමග කොටු වූ සැකකර...,Center
2,01529750-6900-5309-a945-48e448125e25,කන්ජිපානිට සහ ගනේමුල්ලේ සංජීවට රටින් පැනයෑමට උ...,කන්ජිපානිට සහ ගනේමුල්ලේ සංජීවට රටින් පැනයෑමට උ...,Right
3,02136374-ea89-5ad3-9f64-72a24e70ae76,මැණික්වලට අඟරු චෙක් දුන් පහක් අත්අඩංගුවට,මැණික්වලට අඟරු චෙක් දුන් පහක් අත්අඩංගුවට\n\nමැ...,Center
4,021ab263-73e0-51dd-b9eb-3d2ebcafc0a9,හලාවත මත්ද්‍රව්‍ය මෙහෙයුමක් අතරතුර පලා යාමට තැ...,හලාවත මත්ද්‍රව්‍ය මෙහෙයුමක් අතරතුර පලා යාමට තැ...,Right
